# 03. 독립변수 — 2024년 인구 · 건축물 수

**원본**: `Analisis3/06_working_age_population_250m_stable.py`, `06_elderly_population_feature_250m.py`, `05_building_count_feature_250m.py`
세 파일의 로직이 같아서 하나로 합쳤다.

- 방법: 셀 중심점을 SGIS 250m 격자(2024.10)에 공간매칭 (`within`, 실패하면 `intersects`)
- 변수: 20·30·40·50대 인구 → **20~50대 합계**, **고령인구**(대안 변수), **건축물 수**
- 원자료 공란은 0명 처리하고 결측 플래그를 남긴다

**원본 대비 수정**
- 고양·파주 경계에 걸친 SGIS 격자는 두 시 파일에 **값이 나뉘어** 들어있다. 원본은 나중에 처리한 시의 값으로 덮어써서 한쪽이 사라졌다 → 정리본은 **두 시 값을 더한다**
- 맥에서 한글 폴더명(NFD) 때문에 파일을 못 찾는 문제를 고쳤다

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # 정리본 최상위의 config.py

from config import *
setup_notebook()

In [2]:
import unicodedata
import geopandas as gpd

nfc = lambda text: unicodedata.normalize("NFC", text)


def find_shp(folder, keywords):
    folder_path = next(path for path in SGIS_DIR.iterdir() if nfc(path.name) == nfc(folder))
    paths = [path / "vl_blk.shp" for path in folder_path.iterdir()
             if all(nfc(k) in nfc(path.name) for k in keywords) and (path / "vl_blk.shp").exists()]
    assert len(paths) == 1, f"{folder} {keywords}: {paths}"
    return paths[0]


def match(centers, shp_path):
    grid = gpd.read_file(shp_path)[["val", "geometry"]].to_crs("EPSG:5179")
    grid["val"] = pd.to_numeric(grid["val"], errors="coerce")
    joined = gpd.sjoin(centers[["셀 ID", "geometry"]], grid, how="left", predicate="within").drop_duplicates("셀 ID")
    missing = joined["val"].isna()
    if missing.any():  # 중심점이 격자 경계에 놓인 경우
        fallback = gpd.sjoin(centers[centers["셀 ID"].isin(joined.loc[missing, "셀 ID"])][["셀 ID", "geometry"]],
                             grid, how="left", predicate="intersects")
        joined.loc[missing, "val"] = joined.loc[missing, "셀 ID"].map(fallback.drop_duplicates("셀 ID").set_index("셀 ID")["val"])
    return joined.set_index("셀 ID")["val"].dropna()


def build(centers, columns):
    """columns: {결과 컬럼: (고양 폴더, 파주 폴더, 파일명 키워드)} — 두 시 값을 더한다."""
    result = centers[["셀 ID", "중심점 위도", "중심점 경도"]].copy()
    result["시군"] = "미분류"
    for column, (goyang_folder, paju_folder, keyword) in columns.items():
        per_city = []
        for label, folder, city in [("고양", goyang_folder, "고양시"), ("파주", paju_folder, "파주시")]:
            matched = match(centers, find_shp(folder, [keyword, city]))
            per_city.append(matched)
            result.loc[result["셀 ID"].isin(matched.index), "시군"] = label
        result[column] = result["셀 ID"].map(pd.concat(per_city, axis=1).sum(axis=1, min_count=1))
    return result


centers = stable_centers()

In [3]:
age_columns = {f"{age}대 인구": ("고양인구", "파주인구", f"{age}대 인구 수") for age in (20, 30, 40, 50)}
population = build(centers, {**age_columns, "고령인구": ("고양인구", "파주인구", "고령 인구 수")})
population_columns = [*age_columns, "고령인구"]
population["인구 원자료 결측 여부"] = population[population_columns].isna().any(axis=1)
population[population_columns] = population[population_columns].fillna(0).clip(lower=0)
population["20~50대 인구"] = population[list(age_columns)].sum(axis=1)
population = population.sort_values("셀 ID")
population.to_csv(POPULATION_PATH, index=False, encoding="utf-8-sig")

building = build(centers, {"건축물 수": ("건축물수", "건축물수", "건축물 수 합계")})
building["건축물 수 원자료 결측 여부"] = building["건축물 수"].isna()
building["건축물 수"] = building["건축물 수"].fillna(0).clip(lower=0)
building = building.sort_values("셀 ID")
building.to_csv(BUILDING_PATH, index=False, encoding="utf-8-sig")

print(f"인구 결측 플래그: {population['인구 원자료 결측 여부'].sum()}개 | 건축물 결측 플래그: {building['건축물 수 원자료 결측 여부'].sum()}개")
pd.concat([population[["20~50대 인구", "고령인구"]], building[["건축물 수"]]], axis=1).describe().round(1)

인구 결측 플래그: 14개 | 건축물 결측 플래그: 3개


,20~50대 인구,고령인구,건축물 수
count,132.0,132.0,132.0
mean,648.2,202.0,32.8
std,554.8,176.7,31.3
min,0.0,0.0,0.0
25%,101.8,37.0,13.0
50%,551.0,175.5,23.0
75%,1088.2,343.8,38.5
max,2141.0,696.0,187.0


### 원본 결과와 대조 — 달라진 셀은 모두 고양·파주 경계 셀

In [4]:
checks = [
    ("feature_06_20s_50s_population_2024_250m_stable.csv", "20~50대 인구", population),
    ("feature_08_elderly_population_2024_250m_stable.csv", "고령인구", population),
    ("feature_07_building_count_2024_250m_stable.csv", "건축물 수", building),
]
rows = []
for filename, column, new in checks:
    merged = read_csv(ORIGINAL_OUTPUT / filename)[["셀 ID", column]].merge(new[["셀 ID", column]], on="셀 ID", suffixes=(" 원본", " 정리본"))
    diff = merged[merged.iloc[:, 1] != merged.iloc[:, 2]]
    rows += [{"변수": column, "셀 ID": r["셀 ID"], "원본": r.iloc[1], "정리본(두 시 합산)": r.iloc[2]} for _, r in diff.iterrows()]
pd.DataFrame(rows)

,변수,셀 ID,원본,정리본(두 시 합산)
0,20~50대 인구,7883.0,85.0,103.0
1,고령인구,7106.0,0.0,28.0
2,건축물 수,559.0,9.0,46.0
3,건축물 수,7106.0,5.0,23.0
4,건축물 수,7883.0,18.0,34.0
